# Logistics KPI Analytics
### Descriptive Analysis + Root Cause Investigation

**Objective:** Analyze operational KPIs in a smart logistics dataset, identify delay drivers, and produce clear, stakeholder-ready insights.

This notebook is designed in the style of operational analytics work performed in Fulfillment / RAD / GTL environments.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("muted")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["figure.dpi"] = 120

Path("../visualizations").mkdir(exist_ok=True)

In [ ]:
df = pd.read_csv("../data/smart_logistics_dataset.csv")
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df["Is_Delayed"] = df["Logistics_Delay"].astype(bool)

print("Shape:", df.shape)
print("Date range:", df["Timestamp"].min(), "→", df["Timestamp"].max())
df.head()

## 2. Core KPI Summary

In [ ]:
kpi = {
    "Total Records": len(df),
    "Delay Rate (%)": round(df["Logistics_Delay"].mean() * 100, 1),
    "Delayed Shipments": int(df["Logistics_Delay"].sum()),
    "Avg Asset Utilization (%)": round(df["Asset_Utilization"].mean(), 1),
    "Avg Waiting Time (min)": round(df["Waiting_Time"].mean(), 1),
    "Median Waiting Time (min)": round(df["Waiting_Time"].median(), 1),
}

print("=== CORE OPERATIONAL KPIs ===")
print(pd.Series(kpi))

**Analysis – Core KPIs**

- The overall **Delay Rate is high (around 56%)**. This is the most critical metric and should be monitored daily.
- Average **Asset Utilization is ~80%**, which is moderate. There is room to push toward higher utilization targets.
- Average waiting time is approximately **35 minutes**. Waiting time is both a symptom and a contributor to delays.

## 3. Shipment Status & Delay by Traffic

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

status_counts = df["Shipment_Status"].value_counts()
axes[0].pie(status_counts, labels=status_counts.index, autopct="%1.1f%%",
            startangle=90, colors=sns.color_palette("pastel"))
axes[0].set_title("Shipment Status Distribution")

traffic_delay = df.groupby("Traffic_Status")["Logistics_Delay"].mean().sort_values(ascending=False) * 100
sns.barplot(x=traffic_delay.index, y=traffic_delay.values, ax=axes[1], palette="Reds_r")
axes[1].set_title("Delay Rate (%) by Traffic Status")
axes[1].set_ylabel("Delay Rate (%)")
axes[1].set_xlabel("")

plt.tight_layout()
plt.savefig("../visualizations/01_status_and_traffic.png", bbox_inches="tight")
plt.show()

**Analysis – Status & Traffic**

- A large share of records fall into the Delayed category, confirming the high delay rate.
- **Traffic conditions** have a clear relationship with delays. Heavier traffic statuses tend to show higher delay rates.
- Better route planning and traffic-aware scheduling can reduce delays.

## 4. Root Cause Analysis – Delay Reasons

In [ ]:
delayed = df[df["Logistics_Delay"] == 1].copy()
reason_counts = delayed["Logistics_Delay_Reason"].value_counts(dropna=False)
reason_counts.index = reason_counts.index.fillna("Unknown / Not Specified")

plt.figure(figsize=(10, 5))
sns.barplot(x=reason_counts.values, y=reason_counts.index, palette="Reds_r")
plt.title("Root Cause of Logistics Delays")
plt.xlabel("Number of Delayed Shipments")
plt.ylabel("Delay Reason")
plt.tight_layout()
plt.savefig("../visualizations/02_delay_reasons.png", bbox_inches="tight")
plt.show()

print(reason_counts)

**Analysis – Root Causes**

Main delay drivers:

1. **Weather** – highest volume of delays  
2. **Unknown / Not Specified** – large share of missing reason codes  
3. **Traffic** – significant and partially controllable  
4. **Mechanical Failure** – points to maintenance opportunities  

**Key takeaway:** A high number of "Unknown" reasons is a data quality issue. Weather and Traffic should be priority focus areas for process improvement.

## 5. Asset Utilization & Waiting Time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

util_by_asset = df.groupby("Asset_ID")["Asset_Utilization"].mean().sort_values()
sns.barplot(x=util_by_asset.index, y=util_by_asset.values, ax=axes[0], palette="viridis")
axes[0].axhline(util_by_asset.mean(), color="red", linestyle="--", label=f"Average: {util_by_asset.mean():.1f}%")
axes[0].set_title("Average Asset Utilization by Truck")
axes[0].set_ylabel("Utilization (%)")
axes[0].tick_params(axis="x", rotation=45)
axes[0].legend()

sns.boxplot(data=df, x="Is_Delayed", y="Waiting_Time", ax=axes[1], palette=["#22c55e", "#ef4444"])
axes[1].set_title("Waiting Time by Delay Status")
axes[1].set_xlabel("")
axes[1].set_xticklabels(["On Time", "Delayed"])
axes[1].set_ylabel("Waiting Time (min)")

plt.tight_layout()
plt.savefig("../visualizations/03_utilization_and_waiting.png", bbox_inches="tight")
plt.show()

**Analysis – Utilization & Waiting Time**

- Asset utilization is relatively balanced across trucks (mostly high 70s to low 80s).
- Delayed shipments show **higher and more variable waiting times**.
- Waiting time is both a leading indicator and a contributor to delays.

## 6. Inventory vs Demand Forecast

In [ ]:
plt.figure(figsize=(10, 5))
sns.scatterplot(data=df, x="Demand_Forecast", y="Inventory_Level", hue="Is_Delayed",
                palette={True: "#ef4444", False: "#3b82f6"}, alpha=0.65)
plt.plot([100, 300], [100, 300], "k--", label="Perfect Match")
plt.title("Inventory Level vs Demand Forecast")
plt.xlabel("Demand Forecast")
plt.ylabel("Inventory Level")
plt.legend(title="Delayed")
plt.tight_layout()
plt.savefig("../visualizations/04_inventory_vs_demand.png", bbox_inches="tight")
plt.show()

df["Inventory_Gap"] = df["Inventory_Level"] - df["Demand_Forecast"]
print("Average Inventory Gap (Inventory - Forecast):", round(df["Inventory_Gap"].mean(), 1))

**Analysis – Inventory vs Demand**

- Inventory levels and demand forecasts are not tightly aligned.
- Misalignment can create downstream pressure on transportation and increase delays.
- Better coordination between planning and logistics can reduce last-minute pressure.

## 7. Summary of Findings & Recommendations

### Main Findings

1. **Delay Rate is high (~56%)** – primary KPI that needs daily attention and clear escalation thresholds.
2. **Weather and Traffic** are the strongest identifiable drivers of delays.
3. Large number of delays have **Unknown** reasons → data quality issue.
4. **Waiting time** is significantly higher for delayed shipments.
5. Asset utilization is relatively even but still has room for improvement.

### Recommended Actions

| Priority | Action | Expected Impact |
|----------|--------|------------------|
| High | Set Delay Rate threshold (e.g. 45%) and escalate when breached | Faster reaction |
| High | Improve capture of delay reasons (reduce "Unknown") | Better root cause analysis |
| High | Focus weekly reviews on Weather & Traffic delays | Targeted improvement |
| Medium | Track Waiting Time as a leading indicator | Earlier detection of bottlenecks |
| Medium | Review asset utilization and reallocate where needed | Higher fleet efficiency |

---

*This analysis demonstrates structured KPI monitoring, root cause thinking, and clear communication of operational insights — core skills for Process Assistant and Logistics Operations roles.*